# MotionJSON Colab Export and Browser Preview Demo

This notebook complements the UI demo by running the deterministic red-ball extraction, validating the output, exporting a website handoff ZIP, and previewing the generated web runtime in the browser through Colab's port proxy.


## 1. Clone and install MotionJSON


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
import textwrap
import time
import urllib.error
import urllib.request

REPO_URL = "https://github.com/ptse8204/json-animated-video.git"
REPO_DIR = Path("/content/json-animated-video") if Path("/content").exists() else Path("json-animated-video")

def run(cmd, *, cwd=None, check=True, capture=False):
    """Run a command with readable echoing for Colab and local notebooks."""
    if isinstance(cmd, str):
        display_cmd = cmd
        shell = True
    else:
        display_cmd = " ".join(shlex.quote(str(part)) for part in cmd)
        shell = False
    print(f"$ {display_cmd}")
    completed = subprocess.run(
        cmd,
        cwd=cwd,
        check=check,
        shell=shell,
        text=True,
        capture_output=capture,
    )
    if capture:
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr, file=sys.stderr)
    return completed

if not REPO_DIR.exists():
    run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    print(f"Using existing checkout: {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Repository: {Path.cwd()}")
run([sys.executable, "-m", "pip", "install", "-U", "pip"])
run([sys.executable, "-m", "pip", "install", "-e", ".[ui]"])


## 2. Generate, extract, and validate the red-ball demo

This path is CPU-friendly and does not require SAM2, CUDA, detectors, model weights, cloud APIs, or provider credentials.


In [ ]:
run([sys.executable, "examples/make_demo_video.py", "--out", "examples/demo_red_ball.mp4"])
run([
    sys.executable,
    "-m",
    "motionjson.cli",
    "extract",
    "examples/demo_red_ball.mp4",
    "--out",
    "out/demo_red_ball",
    "--mask-provider",
    "threshold",
    "--lower-hsv",
    "0,80,80",
    "--upper-hsv",
    "12,255,255",
    "--sample-fps",
    "12",
    "--max-frames",
    "12",
])
run([sys.executable, "-m", "motionjson.cli", "validate", "out/demo_red_ball"])


## 3. Inspect the generated MotionJSON files


In [ ]:
output_dir = REPO_DIR / "out" / "demo_red_ball"
for name in [
    "scene_graph.json",
    "object_motion.json",
    "web_asset_manifest.json",
    "tracks.json",
    "fallback_diagnostics.json",
    "rights_manifest.json",
]:
    path = output_dir / name
    print(f"{name}: exists={path.exists()} bytes={path.stat().st_size if path.exists() else 0}")

manifest = json.loads((output_dir / "web_asset_manifest.json").read_text())
print("\nManifest summary:")
print(json.dumps({key: manifest.get(key) for key in ["schema", "canvas", "fps", "duration"]}, indent=2))


## 4. Export a website handoff ZIP


In [ ]:
export_zip = output_dir / "exports" / "website_package.zip"
export_zip.parent.mkdir(parents=True, exist_ok=True)
run([
    sys.executable,
    "-m",
    "motionjson.cli",
    "export",
    "out/demo_red_ball",
    "--format",
    "website-zip",
    "--out",
    str(export_zip),
])
print(f"Export ZIP: {export_zip} ({export_zip.stat().st_size:,} bytes)")


## 5. Preview the web runtime in Colab

The preview serves the repository from a local notebook port and opens `examples/plain_js_embed.html` against the generated `web_asset_manifest.json`.


In [ ]:
PREVIEW_PORT = 8010
try:
    preview_process.terminate()  # type: ignore[name-defined]
    preview_process.wait(timeout=5)  # type: ignore[name-defined]
except Exception:
    pass

preview_process = subprocess.Popen(
    [sys.executable, "-m", "http.server", str(PREVIEW_PORT), "--bind", "127.0.0.1"],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
# Give http.server a moment to bind.
time.sleep(2)
preview_path = "/examples/plain_js_embed.html?manifest=/out/demo_red_ball/web_asset_manifest.json"
try:
    from google.colab import output  # type: ignore

    output.serve_kernel_port_as_iframe(PREVIEW_PORT, path=preview_path, height=720)
    print("Open preview in a separate browser tab/window:")
    output.serve_kernel_port_as_window(PREVIEW_PORT, path=preview_path)
except Exception as exc:
    print(f"Colab port proxy unavailable: {exc}")
    print(f"Open locally: http://127.0.0.1:{PREVIEW_PORT}{preview_path}")


## 6. Download the export ZIP


In [ ]:
try:
    from google.colab import files  # type: ignore

    files.download(str(export_zip))
except Exception as exc:
    print(f"Download helper unavailable outside Colab: {exc}")
    print(export_zip)


## 7. Stop the preview server


In [ ]:
try:
    preview_process.terminate()
    preview_process.wait(timeout=10)
    print("Stopped preview server.")
except Exception as exc:
    print(f"Preview process was already stopped or unavailable: {exc}")
